# 03 — Models · Crowdfund Compass (Group 11)
Goal: predict before launch whether a Kickstarter campaign will be funded, compare class models with a baseline, and test the result honestly on future years.

Workflow: 1) Setup → 2) Import and prepare → 3) Baselines and class models (validation year 2024) → 4) Beyond-class model, all-years vs recent model, final test on 2025 and January–June 2026 → 5) Walk-forward, calibration details, support models → 6) Conclusions.

**Time split, never random:** models are trained on earlier launches and judged on later ones. **Headline metric:** AUC *within category* (computed inside each of the 174 categories, then averaged), because our sample over-represents funded campaigns differently in each category.

Runtime: about 20 minutes in Colab. *AI use: code written with Claude (Fable 5.2), 16.11.2026; reviewed by the team.*

## 1) Setup
`features.py` and `modeling.py` are our shared modules (the app uses the same files). Put them in the Drive folder `project`.

In [1]:
import sys, time, json, warnings
warnings.filterwarnings('ignore')
!pip -q install vaderSentiment lightgbm optuna
import numpy as np, pandas as pd, joblib, os
import xgboost as xgb, lightgbm as lgb, optuna
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, confusion_matrix, brier_score_loss, silhouette_score, roc_curve
optuna.logging.set_verbosity(optuna.logging.WARNING); pd.set_option('display.width', 250); pd.set_option('display.max_rows', 200)

# from google.colab import drive; drive.mount('/content/drive')
project = 'drive/My Drive/crowdfund_compass'      # folder with features.py and modeling.py
path = 'drive/My Drive/ks_scrapes/merged_v2'      # folder with the merged parquet parts
out = project + '/models'; os.makedirs(out, exist_ok=True)   # models and result tables are saved here
sys.path.insert(0, project)
import features as F, modeling as M

## 2) Import and prepare
Build the features (notebook 02), then `M.prepare` adds label codes and assigns each campaign to a period: `<=2022`, `2023`, `2024` (validation), `2025` and `2026H1` (test). Launches from July 2026 on are dropped: they ended after the last full crawl.

In [2]:
raw = F.load_merged(path)
X, tables = F.build_features(raw)
X.to_parquet(out + '/features.parquet', index=False)

finished projects after exclusions: 428276


category table rows: 34031


## 3) Baselines and class models, judged on the validation year 2024
Training data here: launches of 2023 (the first year where video and pre-launch page are known for every campaign).

**Text score (class: text mining).** Title and blurb are turned into TF-IDF word weights, and a logistic regression learns which words go with funded campaigns. Each campaign gets one number, `text_score`. Training rows are scored out-of-fold (by a model that never saw them), so the score cannot leak a campaign's own outcome. *Stacking a text model into a second model is beyond class.*

In [3]:
D=M.prepare(pd.read_parquet(out+'/features.parquet')); print(D.period.value_counts().to_dict())
tr=D[D.period=='2023'].copy(); va=D[D.period=='2024'].copy()
tr=tr[tr.has_video.notna()]
t=time.time(); tr['text_score'],(va['text_score'],),_=M.text_score(tr,[va]); print('text score s',round(time.time()-t))
FULL=M.numeric_features(); rows={}
def add(name,p): rows[name]=M.scores(va,p); print(name,{k:round(v,3) for k,v in rows[name].items()},flush=True)

{'<=2022': 340765, '2025': 26533, '2024': 25471, '2023': 18434, '2026H1': 12990}


text score s 7


### 3.1 Baselines
Any model must beat two naive rules: always predict "funded", and predict the funding rate of the campaign's category.

In [4]:
# 1 baselines
add('Baseline: always "funded"',np.full(len(va),tr.success.mean()))
cr=tr.groupby('cat_id').success.mean(); add('Baseline: category funding rate',va.cat_id.map(cr).fillna(tr.success.mean()).values)
add('Text only (TF-IDF + logistic)',1/(1+np.exp(-va.text_score.values)))

Baseline: always "funded" {'AUC within category': 0.5, 'AUC within parent': 0.5, 'AUC overall': 0.5, 'accuracy': 0.711, 'precision': 0.711, 'recall': 1.0, 'F1': 0.831, 'Brier': 0.206}


Baseline: category funding rate {'AUC within category': 0.5, 'AUC within parent': 0.66, 'AUC overall': 0.775, 'accuracy': 0.789, 'precision': 0.808, 'recall': 0.922, 'F1': 0.861, 'Brier': 0.159}


Text only (TF-IDF + logistic) {'AUC within category': 0.685, 'AUC within parent': 0.74, 'AUC overall': 0.796, 'accuracy': 0.77, 'precision': 0.777, 'recall': 0.948, 'F1': 0.854, 'Brier': 0.157}


### 3.2 Class models
Logistic regression, decision tree, random forest and XGBoost. Tree, forest and XGBoost are tuned with `GridSearchCV`; the folds are time-ordered (`TimeSeriesSplit`), so tuning never looks into the future. The depth-3 tree is deliberately small: the app will show its path as a plain explanation.

In [5]:
# 2 class models
tscv=TimeSeriesSplit(n_splits=3)
num=[c for c in FULL if c not in M.LABELS]
pre=ColumnTransformer([('num',make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler()),num),('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=20),M.LABELS)])
logit=make_pipeline(pre,LogisticRegression(C=0.5,max_iter=2000)).fit(tr[FULL],tr.success); add('Logistic regression',logit.predict_proba(va[FULL])[:,1])
Xt=tr[FULL].fillna(-1); Xv=va[FULL].fillna(-1)
g=GridSearchCV(DecisionTreeClassifier(criterion='entropy',random_state=42),{'max_depth':[3,5,7,9,12],'min_samples_leaf':[20,50,100,200]},cv=tscv,scoring='roc_auc').fit(Xt,tr.success)
print('tree best',g.best_params_); add('Decision tree (grid search)',g.predict_proba(Xv)[:,1]); best_tree=g.best_params_
t3=DecisionTreeClassifier(criterion='entropy',max_depth=3,min_samples_leaf=200,random_state=42).fit(Xt,tr.success); add('Decision tree, depth 3 (for the app)',t3.predict_proba(Xv)[:,1])
g=GridSearchCV(RandomForestClassifier(n_estimators=300,n_jobs=2,random_state=42),{'max_depth':[10,None],'min_samples_leaf':[5,20]},cv=tscv,scoring='roc_auc').fit(Xt,tr.success)
print('rf best',g.best_params_); add('Random forest (grid search)',g.predict_proba(Xv)[:,1]); best_rf=g.best_params_
g=GridSearchCV(xgb.XGBClassifier(subsample=.8,colsample_bytree=.8,n_jobs=2,random_state=42),{'max_depth':[3,5,7],'learning_rate':[.03,.1],'n_estimators':[200,400]},cv=tscv,scoring='roc_auc').fit(tr[FULL],tr.success)
print('xgb best',g.best_params_); add('XGBoost (grid search)',g.predict_proba(va[FULL])[:,1]); best_xgb=g.best_params_
def xg(cols,train=tr,w=None): return xgb.XGBClassifier(subsample=.8,colsample_bytree=.8,n_jobs=2,random_state=42,**best_xgb).fit(train[cols],train.success,sample_weight=w)

Logistic regression {'AUC within category': 0.837, 'AUC within parent': 0.867, 'AUC overall': 0.894, 'accuracy': 0.837, 'precision': 0.848, 'recall': 0.94, 'F1': 0.891, 'Brier': 0.119}


tree best {'max_depth': 9, 'min_samples_leaf': 100}


Decision tree (grid search) {'AUC within category': 0.718, 'AUC within parent': 0.812, 'AUC overall': 0.864, 'accuracy': 0.823, 'precision': 0.834, 'recall': 0.937, 'F1': 0.882, 'Brier': 0.128}


Decision tree, depth 3 (for the app) {'AUC within category': 0.663, 'AUC within parent': 0.774, 'AUC overall': 0.841, 'accuracy': 0.805, 'precision': 0.815, 'recall': 0.94, 'F1': 0.873, 'Brier': 0.137}


rf best {'max_depth': None, 'min_samples_leaf': 5}


Random forest (grid search) {'AUC within category': 0.815, 'AUC within parent': 0.867, 'AUC overall': 0.898, 'accuracy': 0.841, 'precision': 0.851, 'recall': 0.94, 'F1': 0.894, 'Brier': 0.113}


xgb best {'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 400}


XGBoost (grid search) {'AUC within category': 0.823, 'AUC within parent': 0.872, 'AUC overall': 0.902, 'accuracy': 0.846, 'precision': 0.861, 'recall': 0.933, 'F1': 0.896, 'Brier': 0.11}


### 3.3 Which feature groups matter?
The same XGBoost is retrained without one group at a time, and with only the six features a typical Kaggle Kickstarter dataset offers (category, country, goal, duration, launch month, title length).

In [6]:
# feature-set variants (same XGBoost)
for nm,cols in [('XGBoost without category norms',M.numeric_features(with_norms=False)),('XGBoost without text score',M.numeric_features(with_text=False)),
                ('XGBoost without video / pre-launch',M.numeric_features(with_presentation=False)),('XGBoost, 6 Kaggle-style features',M.KAGGLE6)]:
    add(nm,xg(cols).predict_proba(va[cols])[:,1])
R=pd.DataFrame(rows).T.round(3); print(R.to_string()); R.to_csv(out+'/stage1_val2024.csv')
joblib.dump({'best_tree':best_tree,'best_rf':best_rf,'best_xgb':best_xgb},out+'/best_params.joblib')

XGBoost without category norms {'AUC within category': 0.84, 'AUC within parent': 0.874, 'AUC overall': 0.901, 'accuracy': 0.838, 'precision': 0.845, 'recall': 0.945, 'F1': 0.893, 'Brier': 0.114}


XGBoost without text score {'AUC within category': 0.816, 'AUC within parent': 0.869, 'AUC overall': 0.899, 'accuracy': 0.841, 'precision': 0.859, 'recall': 0.929, 'F1': 0.893, 'Brier': 0.112}


XGBoost without video / pre-launch {'AUC within category': 0.812, 'AUC within parent': 0.866, 'AUC overall': 0.898, 'accuracy': 0.843, 'precision': 0.859, 'recall': 0.932, 'F1': 0.894, 'Brier': 0.112}


XGBoost, 6 Kaggle-style features {'AUC within category': 0.725, 'AUC within parent': 0.795, 'AUC overall': 0.837, 'accuracy': 0.804, 'precision': 0.809, 'recall': 0.949, 'F1': 0.873, 'Brier': 0.14}


                                      AUC within category  AUC within parent  AUC overall  accuracy  precision  recall     F1  Brier
Baseline: always "funded"                           0.500              0.500        0.500     0.711      0.711   1.000  0.831  0.206
Baseline: category funding rate                     0.500              0.660        0.775     0.789      0.808   0.922  0.861  0.159
Text only (TF-IDF + logistic)                       0.685              0.740        0.796     0.770      0.777   0.948  0.854  0.157
Logistic regression                                 0.837              0.867        0.894     0.837      0.848   0.940  0.891  0.119
Decision tree (grid search)                         0.718              0.812        0.864     0.823      0.834   0.937  0.882  0.128
Decision tree, depth 3 (for the app)                0.663              0.774        0.841     0.805      0.815   0.940  0.873  0.137
Random forest (grid search)                         0.815            

['/home/claude/work/nbrun/best_params.joblib']

### 3.4 Category norms on a second split
The three category-norm features lowered the within-category AUC above. Before dropping them we confirm it on a second split that does not touch the test years (train 2021–22, evaluate 2023).

In [7]:
# category norms on a second, non-test split: train 2021-22 -> 2023 (no presentation features)
D2=D[D.launch_year.isin([2021,2022,2023])].copy(); a=D2[D2.launch_year<=2022].copy(); b=D2[D2.launch_year==2023].copy()
a['text_score'],(b['text_score'],),_=M.text_score(a,[b])
for nm,cols in [('2021-22 -> 2023 with norms',M.numeric_features(with_presentation=False)),('2021-22 -> 2023 without norms',M.numeric_features(with_presentation=False,with_norms=False))]:
    p=xg(cols,a).predict_proba(b[cols])[:,1]; print(nm,round(M.auc_within(b.success,p,b.cat_id),4),round(M.auc_within(b.success,p,b.parent_category),4))

2021-22 -> 2023 with norms 0.8195 0.9125


2021-22 -> 2023 without norms 0.8281 0.8973


**Interpretation:** every class model beats both baselines clearly. Logistic regression, random forest and XGBoost are close to each other; the single tree is weaker. Category norms lower the within-category AUC on both splits, so they are dropped. Title capitalisation was dropped later as well (it made the Pitch score depend on Title Case): 32 features remain in the final model (30 without video and pre-launch page).

## 4) Final models and test
Helper functions: `XG` trains XGBoost with the tuned settings, `logit_pipe` builds the logistic regression pipeline, `with_text` adds the text score without leakage.

In [8]:
BP=joblib.load(out+'/best_params.joblib'); OUT={}
D=M.prepare(pd.read_parquet(out+'/features.parquet'))
FM=M.numeric_features(with_norms=False); FA=M.numeric_features(with_norms=False,with_presentation=False)   # 33 and 31 features
print('features M',len(FM),'A',len(FA))
def XG(tr,cols,w=None,mono=False,**kw):
    p=dict(BP['best_xgb']); p.update(kw)
    if mono: p['monotone_constraints']=M.monotone(cols)   # beyond class: a lower goal can never lower the chance
    return xgb.XGBClassifier(subsample=.8,colsample_bytree=.8,n_jobs=2,random_state=42,**p).fit(tr[cols],tr.success,sample_weight=w)
def logit_pipe(cols):
    num=[c for c in cols if c not in M.LABELS]
    pre=ColumnTransformer([('num',make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler()),num),('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=20),M.LABELS)])
    return make_pipeline(pre,LogisticRegression(C=0.5,max_iter=2000))
def sc(df,p): return {k:round(v,4) for k,v in M.scores(df,p).items()}
def with_text(tr,others):
    tr=tr.copy(); others=[o.copy() for o in others]; oof,sc_,vm=M.text_score(tr,others); tr['text_score']=oof
    for o,s in zip(others,sc_): o['text_score']=s
    return tr,others,vm
P23=D[(D.period=='2023')&D.has_video.notna()]; P24=D[D.period=='2024']; P25=D[D.period=='2025']; P26=D[D.period=='2026H1']

features M 32 A 30


### 4.1 Beyond class: LightGBM tuned with Optuna
*Why:* to check whether a more modern boosting library with a smarter search beats the class method. *In plain words:* LightGBM builds many small decision trees like XGBoost, just faster. Optuna tries settings one after another and learns from earlier tries which region to explore next, instead of testing a fixed grid.

In [9]:
# ---------- (a) validation 2024, trained on 2023: beyond-class model + final candidates
tr,(va,),_=with_text(P23,[P24]); val={}
val['XGBoost (class), 33 features']=sc(va,XG(tr,FM).predict_proba(va[FM])[:,1])
val['Logistic regression, 33 features']=sc(va,logit_pipe(FM).fit(tr[FM],tr.success).predict_proba(va[FM])[:,1])
def objective(t):
    p=dict(n_estimators=t.suggest_int('n_estimators',150,700),learning_rate=t.suggest_float('learning_rate',.01,.15,log=True),num_leaves=t.suggest_int('num_leaves',6,48),
           min_child_samples=t.suggest_int('min_child_samples',20,200),subsample=t.suggest_float('subsample',.6,1),subsample_freq=1,colsample_bytree=t.suggest_float('colsample_bytree',.5,1),reg_lambda=t.suggest_float('reg_lambda',.01,10,log=True))
    s=[]
    for a,b in TimeSeriesSplit(3).split(tr):
        m=lgb.LGBMClassifier(**p,verbose=-1,n_jobs=2,random_state=42).fit(tr[FM].iloc[a],tr.success.iloc[a]); s.append(roc_auc_score(tr.success.iloc[b],m.predict_proba(tr[FM].iloc[b])[:,1]))
    return np.mean(s)
st=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=42)); st.optimize(objective,n_trials=30); BL=st.best_params; print('lgbm best',BL,round(st.best_value,4))
LG=lambda t,cols: lgb.LGBMClassifier(**BL,subsample_freq=1,verbose=-1,n_jobs=2,random_state=42).fit(t[cols],t.success)
val['LightGBM + Optuna (beyond class), 33 features']=sc(va,LG(tr,FM).predict_proba(va[FM])[:,1])
print(pd.DataFrame(val).T.to_string()); OUT['val2024']=val

lgbm best {'n_estimators': 519, 'learning_rate': 0.021553691629987777, 'num_leaves': 38, 'min_child_samples': 27, 'subsample': 0.9955172853889351, 'colsample_bytree': 0.5267694110938989, 'reg_lambda': 1.3495672291568224} 0.9336


                                               AUC within category  AUC within parent  AUC overall  accuracy  precision  recall      F1   Brier
XGBoost (class), 33 features                                0.8402             0.8744       0.9008    0.8382     0.8453  0.9454  0.8925  0.1142
Logistic regression, 33 features                            0.8392             0.8639       0.8900    0.8353     0.8449  0.9409  0.8903  0.1205
LightGBM + Optuna (beyond class), 33 features               0.8446             0.8711       0.8963    0.8423     0.8509  0.9434  0.8948  0.1155


### 4.2 All-years model (A) against recent-years model (M)
A uses every year but cannot use video / pre-launch page (unknown for older projects, and "unknown" depends on the outcome). M uses only launches from 2023 on, with those two features.

In [10]:
# ---------- (b) A (all years, no video/pre-launch) vs M (2023+), on 2024 and 2025
cmp={}
t0=time.time(); trA,(vaA,),_=with_text(D[D.period.isin(['<=2022','2023'])],[P24]); print('text A s',round(time.time()-t0))
for nm,kw in [('A: all years to 2023 (359k), XGBoost tuned on M',{}),('A: same, deeper (depth 6, 600 trees)',dict(max_depth=6,n_estimators=600,learning_rate=.05))]:
    cmp[('2024',nm)]=sc(vaA,XG(trA,FA,**kw).predict_proba(vaA[FA])[:,1])
cmp[('2024','M: 2023 only (18k), without video/pre-launch')]=sc(va,XG(tr,FA).predict_proba(va[FA])[:,1])
cmp[('2024','M: 2023 only (18k), with video/pre-launch')]=val['XGBoost (class), 33 features']
trA2,(teA,te26A),_=with_text(D[D.period.isin(['<=2022','2023','2024'])],[P25,P26])
mA=XG(trA2,FA,max_depth=6,n_estimators=600,learning_rate=.05); cmp[('2025','A: all years to 2024 (385k), deeper')]=sc(teA,mA.predict_proba(teA[FA])[:,1]); cmp[('2025','A: all years to 2024 (385k), XGBoost tuned on M')]=sc(teA,XG(trA2,FA).predict_proba(teA[FA])[:,1])
TR,(te,te26),VM=with_text(pd.concat([P23,P24]),[P25,P26])
cmp[('2025','M: 2023-24 (44k), without video/pre-launch')]=sc(te,XG(TR,FA).predict_proba(te[FA])[:,1])
plain=XG(TR,FM); final=XG(TR,FM,mono=True); p25=final.predict_proba(te[FM])[:,1]; p26=final.predict_proba(te26[FM])[:,1]
cmp[('2025','M: 2023-24 (44k), with video/pre-launch')]=sc(te,plain.predict_proba(te[FM])[:,1])
C=pd.DataFrame(cmp).T; print(C.to_string()); C.to_csv(out+'/A_vs_M.csv'); OUT['A_vs_M']={f'{k[0]} | {k[1]}':v for k,v in cmp.items()}

text A s 137


                                                      AUC within category  AUC within parent  AUC overall  accuracy  precision  recall      F1   Brier
2024 A: all years to 2023 (359k), XGBoost tuned on M               0.8283             0.8641       0.8870    0.8382     0.8548  0.9304  0.8910  0.1159
     A: same, deeper (depth 6, 600 trees)                          0.8347             0.8640       0.8821    0.8448     0.8590  0.9352  0.8955  0.1148
     M: 2023 only (18k), without video/pre-launch                  0.8317             0.8685       0.8958    0.8352     0.8425  0.9448  0.8907  0.1168
     M: 2023 only (18k), with video/pre-launch                     0.8402             0.8744       0.9008    0.8382     0.8453  0.9454  0.8925  0.1142
2025 A: all years to 2024 (385k), deeper                           0.8517             0.8830       0.8990    0.8487     0.8581  0.9422  0.8982  0.1104
     A: all years to 2024 (385k), XGBoost tuned on M               0.8405             0.8741  

### 4.3 Test: 2025 and January–June 2026
Every model is retrained on 2023–2024 with the settings chosen above and evaluated once on the test years.

**Goal rule (beyond class).** The final model is the tuned XGBoost with one extra rule: it may only learn that a higher goal lowers the chance of funding, never the opposite (`monotone_constraints`). *Why:* without it the model said in 6% of campaigns that lowering the goal lowers the chance, which makes no sense as advice. *In plain words:* we tell the model one thing we know about the world, and it costs nothing in accuracy.

In [11]:
# ---------- (c) TEST: every model trained on 2023-24, evaluated once on 2025 and 2026H1
Xt=TR[FM].fillna(-1); test={}
def both(nm,f): test[nm]={'2025':sc(te,f(te)),'2026H1':sc(te26,f(te26))}
both('Baseline: always "funded"',lambda d:np.full(len(d),TR.success.mean()))
cr=TR.groupby('cat_id').success.mean(); both('Baseline: category funding rate',lambda d:d.cat_id.map(cr).fillna(TR.success.mean()).values)
lg=logit_pipe(FM).fit(TR[FM],TR.success); both('Logistic regression',lambda d:lg.predict_proba(d[FM])[:,1])
dt=DecisionTreeClassifier(criterion='entropy',random_state=42,**BP['best_tree']).fit(Xt,TR.success); both('Decision tree',lambda d:dt.predict_proba(d[FM].fillna(-1))[:,1])
d3=DecisionTreeClassifier(criterion='entropy',max_depth=3,min_samples_leaf=200,random_state=42).fit(Xt,TR.success); both('Decision tree, depth 3 (app explanation)',lambda d:d3.predict_proba(d[FM].fillna(-1))[:,1])
rf=RandomForestClassifier(n_estimators=300,n_jobs=2,random_state=42,**BP['best_rf']).fit(Xt,TR.success); both('Random forest',lambda d:rf.predict_proba(d[FM].fillna(-1))[:,1])
both('XGBoost (grid search)',lambda d:plain.predict_proba(d[FM])[:,1])
both('XGBoost + goal rule (final model)',lambda d:final.predict_proba(d[FM])[:,1])
lgbm=LG(TR,FM); both('LightGBM + Optuna (beyond class)',lambda d:lgbm.predict_proba(d[FM])[:,1])
k6=XG(TR,M.KAGGLE6); both('XGBoost, 6 Kaggle-style features',lambda d:k6.predict_proba(d[M.KAGGLE6])[:,1])
T=pd.concat({y:pd.DataFrame({k:v[y] for k,v in test.items()}).T for y in ['2025','2026H1']}); print(T.to_string()); T.to_csv(out+'/test_table.csv'); OUT['test']=test

                                                 AUC within category  AUC within parent  AUC overall  accuracy  precision  recall      F1   Brier
2025   Baseline: always "funded"                              0.5000             0.5000       0.5000    0.7084     0.7084  1.0000  0.8293  0.2066
       Baseline: category funding rate                        0.5000             0.7216       0.8246    0.7896     0.8042  0.9292  0.8622  0.1477
       Logistic regression                                    0.8574             0.8939       0.9191    0.8541     0.8664  0.9388  0.9011  0.1034
       Decision tree                                          0.8225             0.8625       0.8953    0.8345     0.8508  0.9293  0.8883  0.1162
       Decision tree, depth 3 (app explanation)               0.7783             0.8219       0.8622    0.8061     0.8684  0.8560  0.8621  0.1315
       Random forest                                          0.8580             0.8900       0.9168    0.8497     0.8595  0

### 4.4 Calibration (beyond class)
*Why:* the app shows a probability, so 70% must mean about 70 in 100. *In plain words:* Platt scaling is a small correction curve learned on the validation year 2024 that stretches or squeezes the model's scores so that predicted chances match observed frequencies. The Brier score measures the remaining gap (lower is better).

In [12]:
# ---------- (d) calibration: Platt scaling learned on 2024 (model trained on 2023 with the same settings)
twin=XG(tr,FM,mono=True); z=lambda p:np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))
cal=LogisticRegression(C=1e6).fit(z(twin.predict_proba(va[FM])[:,1]).reshape(-1,1),va.success)
c25=cal.predict_proba(z(p25).reshape(-1,1))[:,1]; c26=cal.predict_proba(z(p26).reshape(-1,1))[:,1]
print('Platt a,b',cal.coef_[0][0].round(3),cal.intercept_[0].round(3),'| Brier 2025 raw',round(brier_score_loss(te.success,p25),4),'calibrated',round(brier_score_loss(te.success,c25),4),'| 2026H1 raw',round(brier_score_loss(te26.success,p26),4),'cal',round(brier_score_loss(te26.success,c26),4))
print('mean predicted vs actual 2025: raw',p25.mean().round(3),'cal',c25.mean().round(3),'actual',te.success.mean().round(3),'| 2026H1 raw',p26.mean().round(3),'actual',te26.success.mean().round(3))
bins=pd.cut(p25,np.linspace(0,1,11)); rel=pd.DataFrame({'p':p25,'y':te.success.values}).groupby(bins,observed=True).agg(n=('y','size'),predicted=('p','mean'),actual=('y','mean')).round(3); print(rel.to_string()); rel.to_csv(out+'/reliability_2025.csv')
OUT['calibration']={'brier_raw_2025':float(brier_score_loss(te.success,p25)),'brier_cal_2025':float(brier_score_loss(te.success,c25))}

Platt a,b 0.967 -0.433 | Brier 2025 raw 0.105 calibrated 0.1034 | 2026H1 raw 0.0968 cal 0.0954
mean predicted vs actual 2025: raw 0.74 cal 0.69 actual 0.708 | 2026H1 raw 0.776 actual 0.749
                n  predicted  actual
(0.0, 0.1]   1350      0.053   0.026
(0.1, 0.2]   1244      0.149   0.104
(0.2, 0.3]   1113      0.249   0.189
(0.3, 0.4]   1098      0.349   0.273
(0.4, 0.5]   1166      0.452   0.381
(0.5, 0.6]   1256      0.551   0.456
(0.6, 0.7]   1492      0.652   0.561
(0.7, 0.8]   1818      0.753   0.706
(0.8, 0.9]   2904      0.855   0.808
(0.9, 1.0]  13092      0.967   0.965


### 4.5 Confusion matrix, results by parent category, capped vs other categories

In [13]:
# ---------- (e) confusion matrix, by parent, capped vs rest (2025)
te=te.assign(p=p25); te26=te26.assign(p=p26)
print('confusion 2025 [rows actual failed/funded; cols predicted failed/funded]',confusion_matrix(te.success,(te.p>=.5).astype(int)).tolist())
bp=te.groupby('parent_category').apply(lambda g:pd.Series({'n':len(g),'funded in sample':g.success.mean(),'AUC within category':M.auc_within(g.success,g.p,g.cat_id),'AUC':roc_auc_score(g.success,g.p)})).round(3); print(bp.to_string()); bp.to_csv(out+'/by_parent_2025.csv')
for nm,d in [('2025',te),('2026H1',te26)]:
    for c in [1,0]:
        g=d[d.capped_category==c]; print(nm,'capped' if c else 'other ','n',len(g),'funded',round(g.success.mean(),3),'AUC within category',round(M.auc_within(g.success,g.p,g.cat_id),4))

confusion 2025 [rows actual failed/funded; cols predicted failed/funded] [[4853, 2885], [1118, 17677]]


                      n  funded in sample  AUC within category    AUC
parent_category                                                      
Art              1763.0             0.749                0.870  0.907
Comics           2964.0             0.944                0.901  0.913
Crafts            461.0             0.460                0.854  0.870
Dance              58.0             0.483                0.756  0.774
Design           2416.0             0.810                0.881  0.904
Fashion          1035.0             0.496                0.887  0.908
Film & Video     2593.0             0.478                0.757  0.778
Food              974.0             0.269                0.804  0.826
Games            6108.0             0.840                0.877  0.928
Journalism        138.0             0.370                0.788  0.818
Music            1182.0             0.638                0.810  0.833
Photography       412.0             0.718                0.850  0.862
Publishing       368

2025 other  n 11439 funded 0.608 AUC within category 0.8495
2026H1 capped n 7456 funded 0.822 AUC within category 0.8793


2026H1 other  n 5534 funded 0.651 AUC within category 0.8479


### 4.6 Business metric
A coach can only help part of a cohort. If the coach takes the 30% of campaigns the model rates riskiest, what share of the campaigns that later fail is among them?

In [14]:
# ---------- (f) business metric: the coach helps the riskiest 30%
def reach(d,p,share=.3):
    k=int(len(d)*share); idx=np.argsort(p)[:k]; return d.success.values[idx].__rsub__(1).sum()/(1-d.success).sum()
for nm,d in [('2025',te),('2026H1',te26)]:
    base=d.cat_id.map(cr).fillna(TR.success.mean()).values
    print(nm,'failures reached in riskiest 30%: model',round(reach(d,d.p.values),3),'| category-rate baseline',round(reach(d,base+np.random.default_rng(0).uniform(0,1e-6,len(d))),3),'| 6 features',round(reach(d,k6.predict_proba(d[M.KAGGLE6])[:,1]),3),'| random 0.30 | failures',int((1-d.success).sum()))
    # within-category version: riskiest 30% inside each category
    r=d.groupby('cat_id').p.rank(pct=True); print('   inside each category:',round(((r<=.3)&(d.success==0)).sum()/(d.success==0).sum(),3))

2025 failures reached in riskiest 30%: model 0.759 | category-rate baseline 0.624 | 6 features 0.674 | random 0.30 | failures 7738
   inside each category: 0.523
2026H1 failures reached in riskiest 30%: model 0.81 | category-rate baseline 0.663 | 6 features 0.696 | random 0.30 | failures 3262
   inside each category: 0.521


### 4.7 Level correction to Kickstarter's official rates (beyond class)
*Why:* our sample holds most funded campaigns but well under half of the failed ones, so sample probabilities are too high. *In plain words:* for each parent category we shift the odds by the gap between the funded rate in our sample and Kickstarter's official all-time rate. The official rate is an all-time average and recent rates are higher, so the app shows a range: corrected value (low end) to sample value (high end).

In [15]:
# ---------- (g) level correction to the official rates per parent category
s=TR.groupby('parent_category').success.mean().to_dict()
te['p_sample']=c25; te['p_official']=M.prior_shift(c25,te.parent_category.values,s)
lv=te.groupby('parent_category').agg(n=('success','size'),sample_actual=('success','mean'),mean_p_sample=('p_sample','mean'),mean_p_official=('p_official','mean')).round(3); lv['official']=pd.Series(M.OFFICIAL_RATE); lv['train_sample_rate']=pd.Series(s).round(3); print(lv.to_string()); lv.to_csv(out+'/levels_2025.csv')
print('all: sample mean p',te.p_sample.mean().round(3),'| corrected mean p',te.p_official.mean().round(3),'| median width of range',np.median(te.p_sample-te.p_official).round(3))
# down-weighting alternative: weight failures so the weighted training rate per parent equals the official rate
w=np.where(TR.success==0,TR.parent_category.map(lambda c:(s[c]/(1-s[c]))/(M.OFFICIAL_RATE[c]/(1-M.OFFICIAL_RATE[c]))),1.)
mw=XG(TR,FM,w=w,mono=True); pw=mw.predict_proba(te[FM])[:,1]
print('weighted model: AUC within category',round(M.auc_within(te.success,pw,te.cat_id),4),'| mean p',pw.mean().round(3),'| mean |weighted - shifted|',np.abs(pw-te.p_official).mean().round(3),'| corr',np.corrcoef(pw,te.p_official)[0,1].round(3))

                    n  sample_actual  mean_p_sample  mean_p_official  official  train_sample_rate
parent_category                                                                                  
Art              1763          0.749          0.704            0.569    0.4943              0.732
Comics           2964          0.944          0.908            0.716    0.6884              0.933
Crafts            461          0.460          0.435            0.321    0.2772              0.455
Dance              58          0.483          0.366            0.389    0.6089              0.576
Design           2416          0.810          0.819            0.562    0.4461              0.847
Fashion          1035          0.496          0.519            0.324    0.3139              0.662
Film & Video     2593          0.478          0.459            0.350    0.3826              0.538
Food              974          0.269          0.246            0.216    0.2578              0.306
Games            610

weighted model: AUC within category 0.8594 | mean p 0.6 | mean |weighted - shifted| 0.082 | corr 0.981


### 4.8 Save the models for the app

In [16]:
# ---------- save
joblib.dump({'model':final,'features':FM,'tfidf':VM[0],'text_lr':VM[1],'platt':cal,'sample_rate':s,'tree3':d3,'logit':lg},out+'/models_M.joblib')
te[['id','success','p','p_sample','p_official','cat_id','parent_category','capped_category']].to_parquet(out+'/test2025_pred.parquet'); json.dump(OUT,open(out+'/stage2.json','w'),default=float)
imp=pd.Series(final.get_booster().get_score(importance_type='gain')).sort_values(ascending=False); print((imp/imp.sum()).round(3).head(15).to_string())

text_score              0.310
creator_prior_rate      0.159
prelaunch_activated     0.100
log_goal_usd            0.052
creator_prior_funded    0.047
log_prep_days           0.043
duration_days           0.038
cat_id                  0.033
has_video               0.030
log_goal_per_day        0.021
goal_ratio              0.015
parent_c                0.015
launch_weekday          0.013
creator_prior_n         0.013
name_has_symbol         0.013


## 5) Robustness and support models

In [17]:
BP=joblib.load(out+'/best_params.joblib'); ART=joblib.load(out+'/models_M.joblib'); FM=ART['features']
D=M.prepare(pd.read_parquet(out+'/features.parquet')); D['yr']=np.where(D.period=='2026H1',2026,D.launch_year)

### 5.1 Walk-forward
Train up to year t−1, test on year t, for every year from 2019. Two variants: all earlier years, or only the last two. (30 features: no video / pre-launch page, no text score, so that every year is comparable.)

In [18]:
# ---- 1) walk-forward: train up to year t-1, test on year t (30 features: no video/pre-launch, no text score)
FW=[c for c in M.numeric_features(with_norms=False,with_presentation=False,with_text=False)]
rows=[]
for t in range(2019,2027):
    te=D[D.yr==t]
    for nm,tr in [('all earlier years',D[D.yr<t]),('last 2 years only',D[D.yr.between(t-2,t-1)])]:
        m=xgb.XGBClassifier(subsample=.8,colsample_bytree=.8,n_jobs=2,random_state=42,max_depth=6,n_estimators=400,learning_rate=.05).fit(tr[FW],tr.success)
        p=m.predict_proba(te[FW])[:,1]; rows.append((t if t<2026 else '2026 Jan-Jun',nm,len(tr),len(te),round(te.success.mean(),3),round(M.auc_within(te.success,p,te.cat_id),3),round(M.auc_within(te.success,p,te.parent_category),3),round(roc_auc_score(te.success,p),3)))
W=pd.DataFrame(rows,columns=['test year','training data','train rows','test rows','funded in sample','AUC within category','AUC within parent','AUC overall']); print(W.to_string(index=False)); W.to_csv(out+'/walk_forward.csv',index=False)

   test year     training data  train rows  test rows  funded in sample  AUC within category  AUC within parent  AUC overall
        2019 all earlier years      245846      27956             0.692                0.787              0.902        0.923
        2019 last 2 years only       52022      27956             0.692                0.788              0.916        0.932
        2020 all earlier years      273802      25027             0.735                0.801              0.914        0.935
        2020 last 2 years only       53440      25027             0.735                0.801              0.932        0.948
        2021 all earlier years      298829      22539             0.779                0.802              0.920        0.938
        2021 last 2 years only       52983      22539             0.779                0.802              0.934        0.950
        2022 all earlier years      321368      19397             0.736                0.822              0.897        0.915


### 5.2 Calibration after Platt scaling, and check of the level correction

In [19]:
# ---- 2) calibration detail and threshold
P=pd.read_parquet(out+'/test2025_pred.parquet')
rel=P.groupby(pd.cut(P.p_sample,np.linspace(0,1,11)),observed=True).agg(n=('success','size'),predicted=('p_sample','mean'),actual=('success','mean')).round(3); print('reliability after Platt, 2025'); print(rel.to_string()); rel.to_csv(out+'/reliability_2025_cal.csv')
fpr,tpr,thr=roc_curve(P.success,P.p_sample); j=np.argmax(tpr-fpr); print('Youden J threshold on 2025 (info only):',round(thr[j],3),'tpr',round(tpr[j],3),'fpr',round(fpr[j],3))
# level correction check: weight failures so that each parent matches the official rate, then compare
s=ART['sample_rate']; s25=P.groupby('parent_category').success.mean()
P['w']=np.where(P.success==0,P.parent_category.map(lambda c:(s25[c]/(1-s25[c]))/(M.OFFICIAL_RATE[c]/(1-M.OFFICIAL_RATE[c]))),1.)
chk=P.groupby('parent_category').apply(lambda g:pd.Series({'official':M.OFFICIAL_RATE[g.name],'weighted mean corrected p':np.average(g.p_official,weights=g.w),'mean corrected p':g.p_official.mean(),'mean sample p':g.p_sample.mean()})).round(3)
print(chk.to_string()); print('mean abs gap official vs weighted mean corrected:',(chk.official-chk['weighted mean corrected p']).abs().mean().round(3)); chk.to_csv(out+'/level_check.csv')
print('share of 2025 campaigns: sample p>=0.5',(P.p_sample>=.5).mean().round(3),'| corrected p>=0.5',(P.p_official>=.5).mean().round(3),'| both',((P.p_sample>=.5)&(P.p_official>=.5)).mean().round(3))

reliability after Platt, 2025
                n  predicted  actual
p_sample                            
(0.0, 0.1]   1869      0.052   0.045
(0.1, 0.2]   1549      0.149   0.147
(0.2, 0.3]   1325      0.249   0.263
(0.3, 0.4]   1317      0.351   0.377
(0.4, 0.5]   1302      0.451   0.467
(0.5, 0.6]   1420      0.551   0.570
(0.6, 0.7]   1518      0.651   0.698
(0.7, 0.8]   1994      0.752   0.773
(0.8, 0.9]   3291      0.857   0.883
(0.9, 1.0]  10948      0.961   0.978
Youden J threshold on 2025 (info only): 0.644 tpr 0.84 fpr 0.169
                 official  weighted mean corrected p  mean corrected p  mean sample p
parent_category                                                                      
Art                 0.494                      0.447             0.569          0.704
Comics              0.688                      0.598             0.716          0.908
Crafts              0.277                      0.256             0.321          0.435
Dance               0.609      

### 5.3 Support model: typical money raised (class: linear regression)
Linear regression on log10 of the USD raised. "Error factor ×1.8" means the estimate is typically off by a factor of 1.8 in either direction.

In [20]:
# ---- 3) support model: linear regression on money raised (class)
raw=raw[['id','pledged','static_usd_rate','usd_pledged']]
D=D.merge(raw.assign(raised=raw.usd_pledged.fillna(raw.pledged*raw.static_usd_rate))[['id','raised']],on='id')
D['log_raised']=np.log10(D.raised+1)
tr=D[D.period.isin(['2023','2024'])&D.has_video.notna()].copy(); te=D[D.period=='2025'].copy()
oof,(ts,),_=M.text_score(tr,[te]); tr['text_score']=oof; te['text_score']=ts
num=[c for c in FM if c not in M.LABELS]
def design(d,ref=None):
    Z=d[num].copy(); Z['prior_rate_missing']=Z.creator_prior_rate.isna().astype(int)
    Z=Z.fillna(tr[num].median()); Z=pd.concat([Z,pd.get_dummies(d.parent_category,drop_first=True,dtype=float)],axis=1)
    return Z if ref is None else Z.reindex(columns=ref,fill_value=0)
Zt=design(tr); Zv=design(te,Zt.columns)
for nm,a,b in [('all campaigns',tr,te),('funded campaigns only',tr[tr.success==1],te[te.success==1])]:
    lr=LinearRegression().fit(Zt.loc[a.index],a.log_raised); pr=lr.predict(Zv.loc[b.index]); base=b.cat_id.map(a.groupby('cat_id').log_raised.median()).fillna(a.log_raised.median())
    err=np.abs(pr-b.log_raised); print(f'money raised, {nm}: n_test={len(b)} R2={lr.score(Zv.loc[b.index],b.log_raised):.3f} RMSE(log10)={np.sqrt(((pr-b.log_raised)**2).mean()):.3f} | typical error factor x{10**np.median(err):.2f} | category-median baseline RMSE={np.sqrt(((base-b.log_raised)**2).mean()):.3f} factor x{10**np.median(np.abs(base-b.log_raised)):.2f}')
    if nm.startswith('funded'): joblib.dump({'lr':lr,'cols':list(Zt.columns),'medians':tr[num].median()},out+'/model_raised.joblib')

money raised, all campaigns: n_test=26533 R2=0.397 RMSE(log10)=0.930 | typical error factor x3.41 | category-median baseline RMSE=1.091 factor x3.84
money raised, funded campaigns only: n_test=18795 R2=0.675 RMSE(log10)=0.429 | typical error factor x1.82 | category-median baseline RMSE=0.690 factor x2.74


### 5.4 Campaign archetypes (k-means; clustering is a class topic, its use here is descriptive)
*In plain words:* k-means sorts campaigns into a few groups that look alike on goal, duration, preparation time, creator experience, video, pre-launch page and text length. The silhouette score (0 to 1) says how clearly the groups are separated.

In [21]:
# ---- 4) k-means archetypes (beyond class) on launches 2023-24
A=tr.copy(); A['log_prior']=np.log10(A.creator_prior_n+1)
KF=['log_goal_usd','duration_days','log_prep_days','log_prior','prelaunch_activated','has_video','name_words','blurb_words']
scl=StandardScaler().fit(A[KF]); Z=scl.transform(A[KF]); sm=np.random.default_rng(0).choice(len(Z),6000,replace=False)
for k in range(3,8):
    km=KMeans(k,n_init=5,random_state=42).fit(Z); print('k',k,'silhouette',round(silhouette_score(Z[sm],km.labels_[sm]),3),'inertia',int(km.inertia_))
km=KMeans(4,n_init=10,random_state=42).fit(Z); A['arch']=km.labels_
prof=A.groupby('arch').agg(n=('success','size'),funded_in_sample=('success','mean'),median_goal_usd=('goal_usd','median'),duration=('duration_days','median'),prep_days=('log_prep_days',lambda s:10**s.median()-1),first_timer=('first_timer','mean'),prelaunch=('prelaunch_activated','mean'),video=('has_video','mean'),title_words=('name_words','median')).round(2); print(prof.to_string()); prof.to_csv(out+'/archetypes.csv')
joblib.dump({'scaler':scl,'kmeans':km,'features':KF},out+'/model_archetypes.joblib')

k 3 silhouette 0.17 inertia 256372


k 4 silhouette 0.132 inertia 239019


k 5 silhouette 0.165 inertia 222483


k 6 silhouette 0.14 inertia 209077


k 7 silhouette 0.149 inertia 197357


          n  funded_in_sample  median_goal_usd  duration  prep_days  first_timer  prelaunch  video  title_words
arch                                                                                                           
0     16549              0.82          5000.00     30.00      50.37         0.67       0.99   0.95          7.0
1      9154              0.50          2763.19     30.00       8.02         0.84       0.33   0.00          5.0
2      8689              0.96           600.00     21.00      20.37         0.05       0.73   0.32          7.0
3      9512              0.50          5979.01     30.04      11.57         0.79       0.02   1.00          5.0


['/home/claude/work/nbrun/model_archetypes.joblib']

### 5.5 Similar campaigns (nearest neighbours, beyond class)
*In plain words:* for a new campaign we look up the 10 most similar past campaigns in the same parent category and show how many of them were funded. As a check, that share alone already predicts 2025 outcomes.

In [22]:
# ---- 5) nearest neighbours: similar past campaigns (beyond class)
NF=['log_goal_usd','duration_days','log_prep_days','log_prior','prelaunch_activated','has_video','text_score','first_timer']
te['log_prior']=np.log10(te.creator_prior_n+1); s2=StandardScaler().fit(A[NF]); nb=np.full(len(te),np.nan); pos={i:k for k,i in enumerate(te.index)}
for par,g in A.groupby('parent_category'):
    q=te[te.parent_category==par]
    if len(q)==0: continue
    nn=NearestNeighbors(n_neighbors=10).fit(s2.transform(g[NF])); _,ix=nn.kneighbors(s2.transform(q[NF]))
    nb[[pos[i] for i in q.index]]=g.success.values[ix].mean(axis=1)
te['nb']=nb; print('similar campaigns (10 nearest in the same parent category, 2023-24): funded share of neighbours as a predictor of 2025 outcomes: AUC within category',round(M.auc_within(te.success,te.nb,te.cat_id),3),'| overall',round(roc_auc_score(te.success,te.nb),3))
print(te.groupby(pd.cut(te.nb,[-.01,.2,.4,.6,.8,1])).success.agg(n='size',funded='mean').round(3).to_string())

similar campaigns (10 nearest in the same parent category, 2023-24): funded share of neighbours as a predictor of 2025 outcomes: AUC within category 0.791 | overall 0.876
                  n  funded
nb                         
(-0.01, 0.2]   3046   0.112
(0.2, 0.4]     2108   0.301
(0.4, 0.6]     2719   0.511
(0.6, 0.8]     4237   0.706
(0.8, 1.0]    14423   0.932


## 6) Conclusions
1. All class models beat the baselines by a wide margin; XGBoost with the goal rule is the final model (within-category AUC 0.859 on 2025, 0.866 on January–June 2026; without the rule 0.859 and 0.867).
2. The model family matters little (logistic regression 0.857, random forest 0.858, LightGBM 0.866). The features matter: 6 Kaggle-style features reach 0.753.
3. Two recent years of data do as well as all history inside a category.
4. Probabilities are well calibrated on the sample after Platt scaling, but the sample is too optimistic; the app shows a range down to the level implied by Kickstarter's official rates.
5. If a coach takes the riskiest 30%, 76% of the later failures in our sample are among them (random: 30%).
6. Limits: missing failed campaigns still inflate all scores; associations are not causes.